# Silver - TU PortPS transform (TS-302)

Bronze TransUnion Port/PS responses (`<TNResponse>` XML) -> per-event silver Iceberg table.

Pipeline only: read -> parse XML -> clean -> features -> assemble -> write Iceberg.
Unit tests and data-quality validation live in `TU_step2_silver_tests.ipynb`.

In [1]:
%%pyspark project.spark.compatibility
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, BooleanType,
    TimestampType, DateType, ArrayType, MapType,
)
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, col, udf
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

Executing for connection type: SPARK_GLUE, connection name: project.spark.compatibility
Creating Glue session...
Create session for connection: project.spark.compatibility


'Session ae2xwmv6edkt56-2ae1e6df-ee84-44ba-bfba-a7e59bf33519 has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: project.spark.compatibility.

Connection: project.spark.compatibility | Run start time: 2026-07-29 18:23:31.182186 | Run duration : 0:02:18.539386s.


In [2]:
%%pyspark project.spark.compatibility
spark = (
    SparkSession.builder
    .appName("tu_portps_silver")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.sql.files.maxPartitionBytes", str(1024 * 1024 * 1024))
    .config("spark.hadoop.fs.s3a.connection.maximum", "1000")
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "20")
    .config("spark.hadoop.fs.s3a.retry.limit", "20")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "UTC")  # parse naive strings as UTC, then shift to EST



Connection: project.spark.compatibility | Run start time: 2026-07-29 18:25:49.725869 | Run duration : 0:00:01.139142s.


In [3]:
%%pyspark project.spark.compatibility
# paths / table
BRONZE_PATH          = "s3://enstream-lake-bronze-dev/landing/BNC_processed.csv"  # flat cols + Raw_XML_Response
BRONZE_JOIN_KEY_PATH = "s3://enstream-lake-bronze-dev/landing/BNC_hashed.csv"      # bronze_row_id, acHash, atHash
SILVER_TABLE         = "spark_catalog.trust_score_v1_poc_silver.tu_portps"
SILVER_PATH          = "s3://enstream-lake-silver-dev/poc/tables/dataset=tu_portps/version=1/"


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:25:50.868966 | Run duration : 0:00:00.758362s.


In [4]:
%%pyspark project.spark.compatibility
# standard silver fields
SOURCE_NAME     = "tu_portps"
SCHEMA_VERSION  = 1
RUN_TS          = datetime.now(ZoneInfo("America/Toronto")).replace(tzinfo=None, microsecond=0)


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:25:51.631221 | Run duration : 0:00:01.839005s.


In [5]:
%%pyspark project.spark.compatibility
# EST is a fixed UTC-5 (no DST). Session tz is UTC, so from_utc_timestamp(ts, "EST") shifts -5h.
# Use "America/Toronto" for DST-aware Eastern.
TARGET_TZ = "EST"


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:25:53.474087 | Run duration : 0:00:00.251655s.


In [6]:
%%pyspark project.spark.compatibility
# SPID -> carrier, two levels: brand (*_mno_raw) then big-3 parent (*_mno_std). Current owner only.
# Two known limits (use Abhi's SPID reference table for point-in-time):
#   1. ownership changes over time (Fido->Rogers 2004, MTS->Bell 2017, Shaw/Freedom 2023); this maps to the current owner.
#   2. resellers (Distributel / Primus / Comwave) have ambiguous parents, left independent.

# Level 1: SPID -> brand
SPID_TO_CARRIER = {
    # --- Bell network ---
    "6574": "BELL_MOBILITY", "8050": "BELL", "8051": "BELL", "2933": "BELL_WEST",
    "329A": "ALIANT", "8085": "ALIANT", "8087": "ALIANT", "8089": "ALIANT", "8090": "ALIANT",
    "8088": "MTS", "991B": "MTS", "8239": "TELEBEC", "8092": "NORTHWESTEL", "8228": "NORTHERN_TEL",
    # --- Rogers network ---
    "8821": "ROGERS_WIRELESS", "8377": "ROGERS_CABLE", "743B": "ROGERS_CABLE", "571G": "ROGERS",
    "5643": "FIDO", "8820": "FIDO", "497E": "SHAW",
    # --- Telus network ---
    "8303": "TELUS_MOBILITY", "8301": "TELUS_MOBILITY", "646F": "TELUS_MOBILITY",
    "2243": "TELUS_QC", "8083": "TELUS_QC", "2782": "TELUS_EAST", "8084": "TELUS", "8086": "TELUS",
    # --- non-big-3: Videotron / Quebecor (incl. Freedom) ---
    "328F": "VIDEOTRON", "787G": "VIDEOTRON", "8306": "VIDEOTRON", "383F": "FREEDOM",
    # --- non-big-3: independent / regional ---
    "9868": "SASKTEL", "8091": "SASKTEL", "154E": "IRISTEL", "346J": "IRISTEL",
    "421G": "EASTLINK", "4878": "EASTLINK", "221K": "COGECO", "464D": "FIBERNETICS",
    "081E": "DISTRIBUTEL", "082E": "DISTRIBUTEL", "083E": "DISTRIBUTEL",
    "160G": "COMWAVE", "201A": "PRIMUS", "247F": "XITTEL", "3147": "EXATEL", "4727": "ISP_TELECOM",
    "498E": "PROVINCIAL_TEL", "548H": "TEKSAVVY", "705F": "CITYWEST", "078G": "NAVIGATA",
    "138D": "WIGHTMAN", "139J": "FGLTEL", "141F": "LESNET", "383H": "INNSYSVOICE", "812H": "IXICA",
    "817D": "IVIC", "818D": "COOPTEL", "8202": "PERSONA", "8206": "COCHRANE", "8213": "GUEVREMONT",
    "8221": "MILOT", "8237": "SOGETEL", "8254": "MASKATEL", "8304": "ZAYO",
    "843D": "QC_9164_3122", "920D": "WESTPORT",
}

# Seen only as to_spid, not yet named (waiting on Abhi's reference table):
#   101E 582F 8506 103E 709G 9937 500H 934F 069J 984C 8257 190E 4297 4849

# Level 2: brand -> big-3 parent. Unlisted = independent = *_mno_std null.
CARRIER_PARENT = {
    # -> BELL
    "BELL": "BELL", "BELL_MOBILITY": "BELL", "BELL_WEST": "BELL",
    "ALIANT": "BELL",        # Bell Aliant
    "MTS": "BELL",           # Bell MTS
    "TELEBEC": "BELL",       # Bell via Bell Aliant
    "NORTHWESTEL": "BELL", "NORTHERN_TEL": "BELL",
    # -> ROGERS
    "ROGERS": "ROGERS", "ROGERS_WIRELESS": "ROGERS", "ROGERS_CABLE": "ROGERS",
    "FIDO": "ROGERS",        # Rogers flanker
    "SHAW": "ROGERS",        # Rogers, 2023
    # -> TELUS
    "TELUS": "TELUS", "TELUS_MOBILITY": "TELUS", "TELUS_QC": "TELUS", "TELUS_EAST": "TELUS",
    # non-big-3 brands omitted -> *_mno_std null (Videotron, Freedom, SaskTel, Iristel, Eastlink, etc.)
}
BIG3 = ["BELL", "ROGERS", "TELUS"]


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:25:53.730076 | Run duration : 0:00:00.375269s.


In [7]:
%%pyspark project.spark.compatibility
EVENT_TYPE_MAP = {
    "Inter-SPID Port":              "INTER_SPID_PORT",
    "Intra-SPID Port":              "INTRA_SPID_PORT",
    "NPAC Code Assignment":         "NPAC_CODE_ASSIGNMENT",
    "Return to NPAC Code Assignee": "RETURN_TO_NPAC_CODE_ASSIGNEE",
    "NPAC SPID Update":             "NPAC_SPID_UPDATE",      # administrative SPID change, not a customer port
    "NPAC SPID Migration":          "NPAC_SPID_MIGRATION",   # M&A bulk SPID migration (NANC 323), not a customer port
}
LNP_TYPE_MAP = {"INTER_SPID_PORT": 0, "INTRA_SPID_PORT": 1}  # everything else -> null

# sv_type allowed values (TU API guide, Appendix B "SV Type"). Unknown values go to dq/issues for review.
SV_TYPE_VOCAB = ["Wireline", "Wireless", "VoWIFI", "VoIP", "Pre-Paid Wireless"]

# Silver keeps only these; other events are dropped after from_spid is computed (it uses the dropped rows).
PORT_EVENT_TYPES = ["INTER_SPID_PORT", "INTRA_SPID_PORT"]

OK_CODE = "3000"

# Schema column order (used by the final select and the contract).
SILVER_COLUMNS = [
    "record_id", "phone_number_AC_hash", "phone_number_AT_hash", "event_timestamp", "event_date", "event_type",
    "event_seq", "lnp_type", "is_inter_carrier_port", "from_spid", "from_mno_raw", "from_mno_std",
    "to_spid", "to_mno_raw", "to_mno_std", "from_to_same_carrier", "sv_type", "lrn",
    "correlation_id", "source_name", "schema_version", "ingestion_ts", "source_event_id",
]


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:25:54.109281 | Run duration : 0:00:00.751737s.


In [8]:
%%pyspark project.spark.compatibility
# Read bronze the same way as TU_step1 (\N nulls / quote escaping / multiLine).
bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("nullValue", "\\N")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .csv(BRONZE_PATH)
    .withColumnRenamed("ID", "bronze_row_id")
).cache()
print("bronze rows:", bronze.count())

bronze rows: 13366

Connection: project.spark.compatibility | Run start time: 2026-07-29 18:25:54.864786 | Run duration : 0:00:13.763700s.


In [9]:
%%pyspark project.spark.compatibility
# Read the hashed key file (bronze_row_id, acHash, atHash) -- real upstream sha256 hashes.
bronze_key = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("nullValue", "\\N")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .csv(BRONZE_JOIN_KEY_PATH)
    .withColumnRenamed("ID", "bronze_row_id")
).cache()
print("bronze_key rows:", bronze_key.count())

bronze_key rows: 13366

Connection: project.spark.compatibility | Run start time: 2026-07-29 18:26:08.632646 | Run duration : 0:00:02.224335s.


In [10]:
%%pyspark project.spark.compatibility
# Join acHash/atHash into bronze so they participate in dedup.
bronze = bronze.join(
    bronze_key.select("bronze_row_id", "acHash", "atHash").dropDuplicates(["bronze_row_id"]),
    "bronze_row_id", "left")
print("NULL acHash after join:", bronze.filter(F.col("acHash").isNull()).count())   # expect 0

dedup_cols = [c for c in bronze.columns if c != "bronze_row_id"]   # now includes acHash/atHash

# Collapse duplicate responses (identical row, different bronze_row_id), keep the min bronze_row_id.
bronze_dedup = (
    bronze
    .groupBy(*dedup_cols)
    .agg(F.min("bronze_row_id").alias("bronze_row_id"))
)

# Split on response code.
ok_resp  = bronze_dedup.filter(F.col("API_ResponseCode") == OK_CODE)
non_3000 = bronze_dedup.filter(F.col("API_ResponseCode").isNull() | (F.col("API_ResponseCode") != OK_CODE))
print("ok_3000:", ok_resp.count(), "| non_3000:", non_3000.count())

NULL acHash after join: 0
ok_3000: 9034 | non_3000: 0

Connection: project.spark.compatibility | Run start time: 2026-07-29 18:26:10.861293 | Run duration : 0:00:09.046778s.


## 1. Parse XML

In [11]:
%%pyspark project.spark.compatibility
# Parse one <TNResponse>: capture everything, curate in the select.
parse_struct = StructType([
    StructField("parse_ok",     BooleanType(),                                  True),
    StructField("phone_number", StringType(),                                   True),
    StructField("snapshot",     MapType(StringType(), StringType()),            True),  # non-history leaf path -> value
    StructField("events",       ArrayType(MapType(StringType(), StringType())), True),  # every PortingInfo, all fields
])


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:26:19.912634 | Run duration : 0:00:00.725321s.


In [12]:
%%pyspark project.spark.compatibility
def _local(tag):
    return tag.split("}", 1)[1] if "}" in tag else tag

def parse_tnresponse(xml_str):
    if not xml_str or not xml_str.strip():
        return (False, None, {}, [])
    try:
        root = ET.fromstring(xml_str)
        if "}" in root.tag:            # namespace -> mark bad (guards prod drift)
            return (False, None, {}, [])
        # snapshot: leaf path -> text, excluding PortingHistory
        snapshot = {}
        def walk(el, prefix):
            for ch in el:
                name = _local(ch.tag)
                if name == "PortingHistory":
                    continue
                path = (prefix + "/" + name) if prefix else name
                if len(ch):
                    walk(ch, path)
                else:
                    t = ch.text.strip() if (ch.text and ch.text.strip()) else None
                    if t is not None:
                        snapshot[path] = t
        walk(root, "")
        # events: child fields of every PortingInfo
        events = []
        ph = root.find("PortingHistory")
        if ph is not None:
            for pi in ph.findall("PortingInfo"):
                m = {}
                for ch in pi:
                    t = ch.text.strip() if (ch.text and ch.text.strip()) else None
                    if t is not None:
                        m[_local(ch.tag)] = t
                events.append(m)
        return (True, snapshot.get("Ownership/PhoneNumber"), snapshot, events)
    except Exception:
        return (False, None, {}, [])


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:26:20.641995 | Run duration : 0:00:00.263895s.


In [13]:
%%pyspark project.spark.compatibility
parse_udf = udf(parse_tnresponse, parse_struct)


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:26:20.909504 | Run duration : 0:00:00.763548s.


In [14]:
%%pyspark project.spark.compatibility
parsed_resp = ok_resp.withColumn("p", parse_udf(F.col("Raw_XML_Response"))).cache()  # one row per response


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:26:21.676796 | Run duration : 0:00:01.806487s.


In [15]:
%%pyspark project.spark.compatibility
# Bucket responses: bad XML / no events / has events (response level).
bad_xml    = parsed_resp.filter(~F.col("p.parse_ok"))
good       = parsed_resp.filter(F.col("p.parse_ok"))
no_events  = good.filter(F.col("p.events").isNull() | (F.size("p.events") == 0))
has_events = good.filter(F.size("p.events") > 0)
print("bad_xml:", bad_xml.count(), "| no_events:", no_events.count(), "| has_events:", has_events.count())

bad_xml: 0 | no_events: 215 | has_events: 8819

Connection: project.spark.compatibility | Run start time: 2026-07-29 18:26:23.487417 | Run duration : 0:00:35.346283s.


In [16]:
%%pyspark project.spark.compatibility
# explode to event level; pull the raw_* columns we need from the snapshot + event maps.
parsed = has_events.select(
    "bronze_row_id", "acHash", "atHash",
    F.col("p.phone_number").alias("raw_phone"),
    F.explode("p.events").alias("e"),
).select(
    "bronze_row_id", "acHash", "atHash", "raw_phone",
    F.col("e")["Date"].alias("raw_date"),
    F.col("e")["Event"].alias("raw_event"),
    F.col("e")["Spid"].alias("raw_spid"),
    F.col("e")["LRN"].alias("raw_lrn"),
    F.col("e")["SvType"].alias("raw_svtype"),
)


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:26:58.838351 | Run duration : 0:00:01.021820s.


## 2. Clean data

In [17]:
%%pyspark project.spark.compatibility
# Null-normalize + trim.
def norm(c):
    c = F.trim(c)
    return F.when(c.isin("", "NULL", "\\N"), None).otherwise(c)


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:26:59.864415 | Run duration : 0:00:00.787191s.


In [18]:
%%pyspark project.spark.compatibility
# Phone (kept temporarily).
cleaned = parsed.withColumn("_phone",              norm(F.col("raw_phone")))

# Event-level direct columns.
cleaned = cleaned.withColumn("to_spid", norm(F.col("raw_spid")))
cleaned = cleaned.withColumn("sv_type", norm(F.col("raw_svtype")))

# lrn: strip hyphens.
cleaned = cleaned.withColumn("lrn", F.when(norm(F.col("raw_lrn")).isNotNull(),
                                           F.regexp_replace(norm(F.col("raw_lrn")), "-", "")).otherwise(None))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:00.655125 | Run duration : 0:00:00.386110s.


In [19]:
%%pyspark project.spark.compatibility
# event_type: raw <Event> -> vocab; unknown flagged for DQ.
et_map = F.create_map([x for k, v in EVENT_TYPE_MAP.items() for x in (lit(k), lit(v))])
cleaned = cleaned.withColumn("event_type", et_map[norm(F.col("raw_event"))])
cleaned = cleaned.withColumn("_event_type_unknown",
                             norm(F.col("raw_event")).isNotNull() & F.col("event_type").isNull())


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:01.044452 | Run duration : 0:00:00.749474s.


In [20]:
%%pyspark project.spark.compatibility
# event_timestamp: UTC -> EST (-5h). Single-arg to_timestamp handles variable fractional seconds.
cleaned = cleaned.withColumn("event_timestamp",
                             F.from_utc_timestamp(F.to_timestamp(norm(F.col("raw_date"))), "EST"))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:01.797536 | Run duration : 0:00:00.272757s.


## 3. Features (schema order, one column per cell)

In [21]:
%%pyspark project.spark.compatibility
# phone hashes: use upstream acHash/atHash (no re-hashing). acHash -> account_changes_batch, atHash -> audit_trail_services_3.
featured = (cleaned
    .withColumnRenamed("acHash", "phone_number_AC_hash")
    .withColumnRenamed("atHash", "phone_number_AT_hash"))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:02.074073 | Run duration : 0:00:00.738492s.


In [22]:
%%pyspark project.spark.compatibility
# record_id = sha256(phone_hash | event_ts | to_spid | event_type), 64 hex. Deterministic; the 4-column natural key is the preimage.
featured = featured.withColumn("record_id", F.sha2(F.concat_ws("|",
        F.col("phone_number_AC_hash"),
        F.col("event_timestamp").cast("string"),
        F.coalesce(F.col("to_spid"), lit("")),
        F.coalesce(F.col("event_type"), lit("")),
    ), 256))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:02.816060 | Run duration : 0:00:00.257528s.


In [23]:
%%pyspark project.spark.compatibility
# event_date = date of event_timestamp (partition column, in EST).
featured = featured.withColumn("event_date", F.to_date("event_timestamp"))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:03.076957 | Run duration : 0:00:00.757280s.


In [24]:
%%pyspark project.spark.compatibility
# Per-number window, time ascending. event_seq is computed post-filter (contiguous 1..n); reused by from_spid over the full chain.
w_num = Window.partitionBy("_phone").orderBy(F.col("event_timestamp").asc())


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:03.837764 | Run duration : 0:00:00.724139s.


In [25]:
%%pyspark project.spark.compatibility
# lnp_type = INTER->0 / INTRA->1 / else null.
lnp_map = F.create_map([x for k, v in LNP_TYPE_MAP.items() for x in (lit(k), lit(v))])
featured = featured.withColumn("lnp_type", lnp_map[F.col("event_type")])


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:04.565373 | Run duration : 0:00:00.256919s.


In [26]:
%%pyspark project.spark.compatibility
# is_inter_carrier_port = lnp_type==0 (non-port events -> false).
featured = featured.withColumn("is_inter_carrier_port", F.coalesce(F.col("lnp_type") == lit(0), lit(False)))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:04.825674 | Run duration : 0:00:00.255214s.


In [27]:
%%pyspark project.spark.compatibility
# from_spid = lag(to_spid) over the full chain. Null on: first event, this row NPAC_CODE_ASSIGNMENT, or prev row RETURN (recycle boundary).
featured = featured.withColumn("_prev_to_spid",    F.lag("to_spid").over(w_num))
featured = featured.withColumn("_prev_event_type", F.lag("event_type").over(w_num))
featured = featured.withColumn("from_spid",
    F.when(F.col("_prev_to_spid").isNull(), None)
     .when(F.col("event_type") == lit("NPAC_CODE_ASSIGNMENT"), None)
     .when(F.col("_prev_event_type") == lit("RETURN_TO_NPAC_CODE_ASSIGNEE"), None)
     .otherwise(F.col("_prev_to_spid")))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:05.084300 | Run duration : 0:00:00.708617s.


In [28]:
%%pyspark project.spark.compatibility
# brand + parent lookup maps.
brand_map  = F.create_map([x for k, v in SPID_TO_CARRIER.items() for x in (lit(k), lit(v))])
parent_map = F.create_map([x for k, v in CARRIER_PARENT.items()  for x in (lit(k), lit(v))])
big3_arr   = F.array(*[lit(c) for c in BIG3])


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:05.796428 | Run duration : 0:00:01.816287s.


In [29]:
%%pyspark project.spark.compatibility
# from_mno_raw = from_spid brand lookup.
featured = featured.withColumn("from_mno_raw", brand_map[F.col("from_spid")])


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:07.616376 | Run duration : 0:00:00.246438s.


In [30]:
%%pyspark project.spark.compatibility
# from_mno_std = brand -> big-3 parent, else null.
featured = featured.withColumn("_from_parent", parent_map[F.col("from_mno_raw")])
featured = featured.withColumn("from_mno_std",
                   F.when(F.array_contains(big3_arr, F.col("_from_parent")), F.col("_from_parent")).otherwise(None))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:07.866252 | Run duration : 0:00:00.802960s.


In [31]:
%%pyspark project.spark.compatibility
# to_mno_raw = to_spid looked up to a brand.
featured = featured.withColumn("to_mno_raw", brand_map[F.col("to_spid")])


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:08.672694 | Run duration : 0:00:00.252098s.


In [32]:
%%pyspark project.spark.compatibility
# to_mno_std = brand -> big-3 parent, else null.
featured = featured.withColumn("_to_parent", parent_map[F.col("to_mno_raw")])
featured = featured.withColumn("to_mno_std",
                   F.when(F.array_contains(big3_arr, F.col("_to_parent")), F.col("_to_parent")).otherwise(None))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:08.928085 | Run duration : 0:00:00.719279s.


In [33]:
%%pyspark project.spark.compatibility
# from_to_same_carrier: 1 same brand, 0 different, null if either side null.
featured = featured.withColumn("from_to_same_carrier",
    F.when(F.col("from_mno_raw").isNull() | F.col("to_mno_raw").isNull(), None)
     .otherwise((F.col("from_mno_raw") == F.col("to_mno_raw")).cast("int")))


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:09.650840 | Run duration : 0:00:00.252762s.


In [34]:
%%pyspark project.spark.compatibility
# Port-only filter: keep INTER/INTRA, drop the rest (full-chain features already computed above).
_rows_before = featured.count()
featured = featured.filter(F.col("event_type").isin(*PORT_EVENT_TYPES))
_rows_after = featured.count()
print("port filter: kept", _rows_after, "of", _rows_before, "event rows;", _rows_before - _rows_after, "dropped non-port event rows")

# event_seq over the retained rows -> contiguous 1..n per number.
featured = featured.withColumn("event_seq", F.row_number().over(w_num))

port filter: kept 8894 of 20276 event rows; 11382 dropped non-port event rows
Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:09.907014 | Run duration : 0:00:03.868191s.


In [35]:
%%pyspark project.spark.compatibility
# Constants / lineage columns.
featured = featured.withColumn("source_name",     lit(SOURCE_NAME))
featured = featured.withColumn("schema_version",  lit(SCHEMA_VERSION).cast("int"))
featured = featured.withColumn("ingestion_ts",    lit(RUN_TS).cast("timestamp"))    # late-event watermark
featured = featured.withColumn("correlation_id",  F.lit(None).cast("string"))        # null placeholder
featured = featured.withColumn("source_event_id", F.col("bronze_row_id").cast("long"))  # bronze source id


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:13.779281 | Run duration : 0:00:02.509903s.


## 4. Assemble + write

In [36]:
%%pyspark project.spark.compatibility
silver = featured.select(*SILVER_COLUMNS)

# Contract schema.
EXPECTED_SCHEMA = StructType([
    StructField("record_id",                  StringType(),    False),
    StructField("phone_number_AC_hash",       StringType(),    False),
    StructField("phone_number_AT_hash",       StringType(),    False),
    StructField("event_timestamp",            TimestampType(), False),
    StructField("event_date",                 DateType(),      False),
    StructField("event_type",                 StringType(),    False),
    StructField("event_seq",                  IntegerType(),   False),
    StructField("lnp_type",                   IntegerType(),   True),
    StructField("is_inter_carrier_port",      BooleanType(),   False),
    StructField("from_spid",                  StringType(),    True),
    StructField("from_mno_raw",               StringType(),    True),
    StructField("from_mno_std",               StringType(),    True),
    StructField("to_spid",                    StringType(),    False),
    StructField("to_mno_raw",                 StringType(),    True),
    StructField("to_mno_std",                 StringType(),    True),
    StructField("from_to_same_carrier",       IntegerType(),   True),
    StructField("sv_type",                    StringType(),    True),
    StructField("lrn",                        StringType(),    True),
    StructField("correlation_id",             StringType(),    True),
    StructField("source_name",                StringType(),    False),
    StructField("schema_version",             IntegerType(),   False),
    StructField("ingestion_ts",               TimestampType(), False),
    StructField("source_event_id",            LongType(),      False),
])

# Compare (name, type) only; nullable is not checked.
def _sig(s): return [(f.name, f.dataType.simpleString()) for f in s.fields]
assert _sig(silver.schema) == _sig(EXPECTED_SCHEMA), \
    ("schema drift:", [x for x in _sig(silver.schema) if x not in _sig(EXPECTED_SCHEMA)],
                      [x for x in _sig(EXPECTED_SCHEMA) if x not in _sig(silver.schema)])
print("schema OK,", len(SILVER_COLUMNS), "cols")

schema OK, 23 cols

Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:16.294050 | Run duration : 0:00:00.628828s.


In [37]:
%%pyspark project.spark.compatibility
# Silver write pattern: CREATE TABLE DDL + overwritePartitions.
import boto3
from urllib.parse import urlparse

def wipe_s3_prefix(s3_uri):
    # Empty an S3 prefix before rebuild (old files/metadata linger otherwise).
    p = urlparse(s3_uri); bucket, prefix = p.netloc, p.path.lstrip("/")
    s3 = boto3.client("s3"); n = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        objs = [{"Key": o["Key"]} for o in page.get("Contents", [])]
        if objs:
            s3.delete_objects(Bucket=bucket, Delete={"Objects": objs}); n += len(objs)
    print("deleted", n, "objects under s3://" + bucket + "/" + prefix)
    chk = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=5)
    if "Contents" in chk:
        print("WARN remaining", len(chk["Contents"]), "objects:")
        for o in chk["Contents"][:5]:
            print("  ", o["Key"])
    else:
        print("OK S3 folder is empty")

def write_iceberg(df, table_name, s3_location, schema_ddl, format_version="3"):
    # Create table + write. schema_ddl includes the partition column event_date.
    ddl = (f"CREATE TABLE IF NOT EXISTS {table_name} (\n{schema_ddl}\n) "
           f"USING iceberg PARTITIONED BY (event_date) "
           f"LOCATION '{s3_location}' "
           f"TBLPROPERTIES ('format-version' = '{format_version}')")
    spark.sql(ddl)
    df.writeTo(table_name).overwritePartitions()

def verify_iceberg(table_name):
    print("=== A. SHOW CREATE TABLE ===")
    spark.sql(f"SHOW CREATE TABLE {table_name}").show(truncate=False)
    print("=== B. file paths ===")
    spark.sql(f"SELECT file_path FROM {table_name}.files LIMIT 5").show(truncate=False)
    print("=== C. file count ===")
    spark.sql(f"SELECT COUNT(*) AS file_count FROM {table_name}.files").show()
    print("=== D. partitions ===")
    spark.sql(f"SELECT partition, spec_id, record_count FROM {table_name}.partitions ORDER BY partition LIMIT 10").show(truncate=False)
    print("row count:", spark.table(table_name).count())


Connection: project.spark.compatibility | Run start time: 2026-07-29 18:27:16.926749 | Run duration : 0:00:00.738938s.


In [38]:
%%pyspark project.spark.compatibility
# DDL from EXPECTED_SCHEMA (includes the partition column event_date).
_SQL = {"string": "STRING", "timestamp": "TIMESTAMP", "date": "DATE",
        "int": "INT", "bigint": "BIGINT", "boolean": "BOOLEAN"}
SILVER_DDL = ",\n    ".join(f"{f.name} {_SQL[f.dataType.simpleString()]}" for f in EXPECTED_SCHEMA.fields)
print(SILVER_DDL)

# Clear first: catalog drop + S3 wipe (else CREATE TABLE IF NOT EXISTS reuses the old schema).
spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
wipe_s3_prefix(SILVER_PATH)

# Write sorted; overwritePartitions is idempotent.
write_iceberg(
    df=silver.sortWithinPartitions("phone_number_AC_hash", "event_timestamp"),
    table_name=SILVER_TABLE,
    s3_location=SILVER_PATH,
    schema_ddl=SILVER_DDL,
    format_version="3",
)

# Verify.
verify_iceberg(SILVER_TABLE)

record_id STRING,
    phone_number_AC_hash STRING,
    phone_number_AT_hash STRING,
    event_timestamp TIMESTAMP,
    event_date DATE,
    event_type STRING,
    event_seq INT,
    lnp_type INT,
    is_inter_carrier_port BOOLEAN,
    from_spid STRING,
    from_mno_raw STRING,
    from_mno_std STRING,
    to_spid STRING,
    to_mno_raw STRING,
    to_mno_std STRING,
    from_to_same_carrier INT,
    sv_type STRING,
    lrn STRING,
    correlation_id STRING,
    source_name STRING,
    schema_version INT,
    ingestion_ts TIMESTAMP,
    source_event_id BIGINT
deleted 4307 objects under s3://enstream-lake-silver-dev/poc/tables/dataset=tu_portps/version=1/
OK S3 folder is empty
=== A. SHOW CREATE TABLE ===
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------